In [ ]:
import warp as wp, numpy as np
from socu._mykernel import MyKernel, device_ptr
wp.init()
k=MyKernel()
B,M,n=1,2,3
E=wp.from_numpy(np.random.randn(B*M,n,n).astype(np.float32), dtype=wp.float32, device='cuda')
D=wp.from_numpy(np.random.randn(B*M,n,n).astype(np.float32), dtype=wp.float32, device='cuda')
s=k.create_stream()
k.syrk_update(device_ptr(E), device_ptr(D), B,M,n, s)
k.stream_synchronize(s)
k.destroy_stream(s)
print('ok', float(D.numpy()[0,0,0]))
display(D.numpy())

In [ ]:
import warp as wp, numpy as np
from socu._mykernel import MyKernel, device_ptr

wp.init()
k = MyKernel()

rng = np.random.default_rng(42)
B, M, n = 100, 21, 400
batch_size = B * M

# 用 float64 生成/计算参考值（高精度）
E0_64 = rng.standard_normal((batch_size, n, n), dtype=np.float64)
D0_64 = rng.standard_normal((batch_size, n, n), dtype=np.float64)

# 当前 CUDA kernel 的接口是 float*，所以送入 GPU 的仍需是 float32
E0 = E0_64.astype(np.float32)
D0 = D0_64.astype(np.float32)

E = wp.from_numpy(E0, dtype=wp.float32, device="cuda")
D = wp.from_numpy(D0, dtype=wp.float32, device="cuda")

s = k.create_stream()
k.syrk_update(device_ptr(E), device_ptr(D), B, M, n, s)
k.stream_synchronize(s)
k.destroy_stream(s)

D_cuda = D.numpy()
print("ok", float(D_cuda[0, 0, 0]))

# NumPy float64 参考：批量矩阵乘法 (batch, n, n) @ (batch, n, n).T
D_ref64 = D0_64 - (E0_64 @ np.swapaxes(E0_64, -1, -2))
D_ref32 = D_ref64.astype(np.float32)

# 对 float32 参考做 allclose（更公平：同一 dtype）
diff32 = D_cuda - D_ref32
max_abs_diff32 = np.max(np.abs(diff32))
max_rel_diff32 = np.max(np.abs(diff32) / (np.abs(D_ref32) + 1e-12))
ok32 = np.allclose(D_cuda, D_ref32, atol=1e-5, rtol=1e-5)

# 同时报告相对 float64 参考的误差（反映 float32 舍入/累加误差）
diff64 = D_cuda.astype(np.float64) - D_ref64
max_abs_diff64 = np.max(np.abs(diff64))
max_rel_diff64 = np.max(np.abs(diff64) / (np.abs(D_ref64) + 1e-18))

print("allclose vs D_ref32:", ok32)
print("max_abs_err vs D_ref32:", float(max_abs_diff32))
print("max_rel_err vs D_ref32:", float(max_rel_diff32))
print("max_abs_err vs D_ref64:", float(max_abs_diff64))
print("max_rel_err vs D_ref64:", float(max_rel_diff64))
print("sample D_cuda[0,0,0]:", float(D_cuda[0, 0, 0]))
print("sample D_ref32[0,0,0]:", float(D_ref32[0, 0, 0]))
print("sample D_ref64[0,0,0]:", float(D_ref64[0, 0, 0]))

# 需要 notebook 环境才有 display；不确定环境时用 print 即可
try:
    from IPython.display import display
    display(D_ref64)
except Exception:
    print(D_ref64[0])

In [2]:
# Benchmark: CUDA (events) vs NumPy (perf_counter)
# Tips for fairness: restart kernel before timing; do warmup; avoid including D.numpy() in GPU timing.

import time
import numpy as np
import warp as wp
from socu._mykernel import MyKernel, device_ptr

wp.init()
k = MyKernel()

# Choose a size that is feasible for both GPU and CPU timing.
# Your earlier (B=100,M=21,n=400) is huge for CPU; start smaller and scale up.
B, M, n = 10, 10, 256
batch_size = B * M
rng = np.random.default_rng(123)

# Host data (float32 to match the CUDA API)
E0 = rng.standard_normal((batch_size, n, n), dtype=np.float32)
D0 = rng.standard_normal((batch_size, n, n), dtype=np.float32)

# Device data
E = wp.from_numpy(E0, dtype=wp.float32, device="cuda")
D = wp.from_numpy(D0, dtype=wp.float32, device="cuda")

# -------------------
# CUDA timing (kernel only)
# -------------------
stream = k.create_stream()

# Warmup (important: CUDA context, caches, etc.)
for _ in range(5):
    k.syrk_update(device_ptr(E), device_ptr(D), B, M, n, stream)
k.stream_synchronize(stream)

use_events = hasattr(k, "create_event") and callable(getattr(k, "create_event", None))
if use_events:
    try:
        start_ev = k.create_event()
        end_ev = k.create_event()
        iters = 30
        # Record start/end around the kernel launches on the SAME stream
        k.event_record(start_ev, stream)
        for _ in range(iters):
            k.syrk_update(device_ptr(E), device_ptr(D), B, M, n, stream)
        k.event_record(end_ev, stream)
        k.event_synchronize(end_ev)
        total_ms = k.event_elapsed_ms(start_ev, end_ev)
        print(f"CUDA events: total {total_ms:.3f} ms for {iters} iters; avg {total_ms/iters:.4f} ms/iter")
        k.destroy_event(start_ev)
        k.destroy_event(end_ev)
    except Exception as e:
        print("CUDA event timing not available (rebuild mykernel.dll to enable). Falling back.")
        print("Reason:", repr(e))
        use_events = False

if not use_events:
    # Fallback: wall time + stream sync (includes some launch overhead; still workable).
    iters = 30
    t0 = time.perf_counter()
    for _ in range(iters):
        k.syrk_update(device_ptr(E), device_ptr(D), B, M, n, stream)
    k.stream_synchronize(stream)
    t1 = time.perf_counter()
    total_ms = (t1 - t0) * 1e3
    print(f"CUDA perf_counter+sync: total {total_ms:.3f} ms for {iters} iters; avg {total_ms/iters:.4f} ms/iter")

k.destroy_stream(stream)

# -------------------
# NumPy timing (CPU)
# -------------------
def numpy_syrk_update_inplace(E_batch: np.ndarray, D_batch: np.ndarray) -> None:
    # D -= E @ E.T for each batch element, in-place
    for i in range(E_batch.shape[0]):
        # Force float32 matmul for apples-to-apples with CUDA output dtype
        D_batch[i] -= E_batch[i] @ E_batch[i].T

# Warmup NumPy/BLAS
D_cpu = D0.copy()
numpy_syrk_update_inplace(E0, D_cpu)

# Time NumPy
D_cpu = D0.copy()
iters = 30  # CPU is slower; keep small or reduce batch_size/n
t0 = time.perf_counter()
for _ in range(iters):
    numpy_syrk_update_inplace(E0, D_cpu)
t1 = time.perf_counter()
total_ms = (t1 - t0) * 1e3
print(f"NumPy perf_counter: total {total_ms:.3f} ms for {iters} iters; avg {total_ms/iters:.3f} ms/iter")

print("Note: NumPy timing depends heavily on BLAS + thread count; for repeatability set OMP/MKL/OPENBLAS threads to 1 and restart kernel.")

CUDA events: total 2389.724 ms for 30 iters; avg 79.6575 ms/iter
NumPy perf_counter: total 814.581 ms for 30 iters; avg 27.153 ms/iter
Note: NumPy timing depends heavily on BLAS + thread count; for repeatability set OMP/MKL/OPENBLAS threads to 1 and restart kernel.
